# Query Iceberg Tables via Trino + Polaris REST Catalog

Reads `demo.users` and `demo.transactions` written by the Flink job, using
**Trino** (catalog `iceberg`) connected to Apache Polaris.

**Prerequisites:** the Docker Compose stack must be running (`docker compose up -d`).

In [2]:
import trino
import pandas as pd

conn = trino.dbapi.connect(
    host="trino-coordinator",
    port=8080,
    user="jupyter",
    catalog="iceberg",
    schema="demo",
)

def query(sql: str) -> pd.DataFrame:
    """Run a Trino SQL query and return a DataFrame."""
    cur = conn.cursor()
    cur.execute(sql)
    rows = cur.fetchall()
    cols = [d[0] for d in cur.description] if cur.description else []
    return pd.DataFrame(rows, columns=cols)

query("SHOW TABLES")

,Table
0,transactions
1,users


---
## Users

In [3]:
# All users
query("SELECT * FROM iceberg.demo.users ORDER BY created_at desc")

,user_id,name,email,country,created_at
0,user-050,Frank Brown,frank.brown50@example.com,FR,2026-05-21 17:30:46.878
1,user-049,Dave Robinson,dave.robinson49@example.com,US,2026-05-21 17:30:46.878
2,user-048,Hank Taylor,hank.taylor48@example.com,FR,2026-05-21 17:30:46.878
3,user-047,Grace Roberts,grace.roberts47@example.com,CA,2026-05-21 17:30:46.878
4,user-046,Bob Lewis,bob.lewis46@example.com,JP,2026-05-21 17:30:46.878
...,...,...,...,...,...
295,user-031,Tara Thomas,tara.thomas31@example.com,US,2026-05-14 18:26:54.740
296,user-012,Jack Johnson,jack.johnson12@example.com,CA,2026-05-14 18:26:54.740
297,user-013,Sam Jones,sam.jones13@example.com,FR,2026-05-14 18:26:54.740
298,user-014,Mia Roberts,mia.roberts14@example.com,BR,2026-05-14 18:26:54.740


In [4]:
# Summary stats
query("""
    SELECT
        count(*)                AS total_users,
        count(DISTINCT country) AS unique_countries,
        min(created_at)         AS earliest_signup,
        max(created_at)         AS latest_signup
    FROM iceberg.demo.users
""")

,total_users,unique_countries,earliest_signup,latest_signup
0,300,8,2026-05-14 18:26:54.740,2026-05-21 17:30:46.878


In [5]:
# Users per country
query("""
    SELECT
        country,
        count(*) AS user_count
    FROM iceberg.demo.users
    GROUP BY country
    ORDER BY user_count DESC
""")

,country,user_count
0,US,49
1,AU,43
2,FR,41
3,DE,38
4,JP,37
5,BR,34
6,CA,31
7,GB,27


---
## Transactions

In [6]:
# Latest 20 transactions
query("""
    SELECT *
    FROM iceberg.demo.transactions
    ORDER BY event_time DESC
    LIMIT 20
""")

,transaction_id,user_id,amount,currency,type,status,event_time
0,txn-011249,user-014,1553.03,AUD,REFUND,COMPLETED,2026-05-21 19:05:44.643
1,txn-011248,user-001,1831.87,USD,WITHDRAWAL,COMPLETED,2026-05-21 19:05:44.141
2,txn-011247,user-049,487.16,AUD,REFUND,FAILED,2026-05-21 19:05:43.639
3,txn-011246,user-008,428.99,USD,TRANSFER,REVERSED,2026-05-21 19:05:43.139
4,txn-011245,user-017,225.97,EUR,PURCHASE,COMPLETED,2026-05-21 19:05:42.637
5,txn-011244,user-037,145.06,CAD,REFUND,FAILED,2026-05-21 19:05:42.136
6,txn-011243,user-040,1113.48,AUD,TRANSFER,PENDING,2026-05-21 19:05:41.634
7,txn-011242,user-021,1790.36,GBP,TRANSFER,COMPLETED,2026-05-21 19:05:41.132
8,txn-011241,user-011,992.39,EUR,DEPOSIT,COMPLETED,2026-05-21 19:05:40.631
9,txn-011240,user-031,883.17,CAD,REFUND,REVERSED,2026-05-21 19:05:40.126


In [7]:
# Summary stats
query("""
    SELECT
        count(*)                AS total_transactions,
        count(DISTINCT user_id) AS unique_users,
        round(sum(amount), 2)   AS total_volume,
        round(avg(amount), 2)   AS avg_amount,
        min(event_time)         AS earliest,
        max(event_time)         AS latest
    FROM iceberg.demo.transactions
""")

,total_transactions,unique_users,total_volume,avg_amount,earliest,latest
0,112307,50,1.120865e+08,998.04,2026-05-14 18:26:55.065,2026-05-21 19:05:44.643


In [8]:
# Volume by transaction type
query("""
    SELECT
        type,
        count(*)              AS tx_count,
        round(sum(amount), 2) AS total_amount
    FROM iceberg.demo.transactions
    GROUP BY type
    ORDER BY tx_count DESC
""")

,type,tx_count,total_amount
0,PURCHASE,22679,22687113.75
1,REFUND,22439,22470411.91
2,WITHDRAWAL,22434,22307266.47
3,TRANSFER,22403,22355744.10
4,DEPOSIT,22352,22265970.20


In [9]:
# Breakdown by status
query("""
    SELECT
        status,
        count(*)              AS tx_count,
        round(sum(amount), 2) AS total_amount
    FROM iceberg.demo.transactions
    GROUP BY status
    ORDER BY tx_count DESC
""")

,status,tx_count,total_amount
0,PENDING,28363,28361460.07
1,REVERSED,28086,27908241.51
2,FAILED,28064,28062939.01
3,COMPLETED,27853,27811014.91


In [10]:
# Volume by currency
query("""
    SELECT
        currency,
        count(*)              AS tx_count,
        round(sum(amount), 2) AS total_amount
    FROM iceberg.demo.transactions
    GROUP BY currency
    ORDER BY total_amount DESC
""")

,currency,tx_count,total_amount
0,AUD,22556,22637153.76
1,GBP,22456,22495922.23
2,CAD,22519,22446453.06
3,EUR,22479,22445765.14
4,USD,22356,22118361.31


In [11]:
# Top 10 users by transaction volume
query("""
    SELECT
        user_id,
        count(*)              AS tx_count,
        round(sum(amount), 2) AS total_spent
    FROM iceberg.demo.transactions
    GROUP BY user_id
    ORDER BY total_spent DESC
    LIMIT 10
""")

,user_id,tx_count,total_spent
0,user-012,2318,2340917.53
1,user-006,2328,2333632.06
2,user-019,2271,2314723.34
3,user-024,2313,2309973.01
4,user-037,2298,2305677.83
5,user-044,2299,2304262.31
6,user-039,2264,2303284.58
7,user-008,2309,2298256.86
8,user-005,2271,2297901.16
9,user-030,2310,2297325.21


---
## Cross-table: join users ↔ transactions

In [12]:
# Enrich transactions with user name and country
query("""
    SELECT
        t.transaction_id,
        u.name,
        u.country,
        t.type,
        t.currency,
        t.amount,
        t.status,
        t.event_time
    FROM iceberg.demo.transactions AS t
    JOIN iceberg.demo.users        AS u ON t.user_id = u.user_id
    ORDER BY t.event_time DESC
    LIMIT 20
""")

,transaction_id,name,country,type,currency,amount,status,event_time
0,txn-011308,Sam Wilson,GB,DEPOSIT,AUD,937.04,COMPLETED,2026-05-21 19:06:14.272
1,txn-011308,Jack Clark,GB,DEPOSIT,AUD,937.04,COMPLETED,2026-05-21 19:06:14.272
2,txn-011308,Quinn Clark,FR,DEPOSIT,AUD,937.04,COMPLETED,2026-05-21 19:06:14.272
3,txn-011308,Olivia Martinez,BR,DEPOSIT,AUD,937.04,COMPLETED,2026-05-21 19:06:14.272
4,txn-011308,Frank Brown,AU,DEPOSIT,AUD,937.04,COMPLETED,2026-05-21 19:06:14.272
5,txn-011308,Karen Roberts,US,DEPOSIT,AUD,937.04,COMPLETED,2026-05-21 19:06:14.272
6,txn-011307,Leo Martinez,FR,WITHDRAWAL,CAD,286.80,COMPLETED,2026-05-21 19:06:13.771
7,txn-011307,Quinn Brown,US,WITHDRAWAL,CAD,286.80,COMPLETED,2026-05-21 19:06:13.771
8,txn-011307,Leo Brown,FR,WITHDRAWAL,CAD,286.80,COMPLETED,2026-05-21 19:06:13.771
9,txn-011307,Leo Brown,AU,WITHDRAWAL,CAD,286.80,COMPLETED,2026-05-21 19:06:13.771


In [13]:
# Total transaction volume per country
query("""
    SELECT
        u.country,
        count(*)                AS tx_count,
        round(sum(t.amount), 2) AS total_volume
    FROM iceberg.demo.transactions AS t
    JOIN iceberg.demo.users        AS u ON t.user_id = u.user_id
    GROUP BY u.country
    ORDER BY total_volume DESC
""")

,country,tx_count,total_volume
0,US,109790,1.096445e+08
1,AU,96766,9.638410e+07
2,FR,92331,9.228621e+07
3,DE,85301,8.514750e+07
4,JP,83082,8.285251e+07
5,BR,76276,7.605468e+07
6,CA,69590,6.933387e+07
7,GB,61060,6.115853e+07
